### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="consumer_complaints_1m",
    version_from_unique_name="consumer_complaints",
    version_comment="""
We use the last 3 months as test data randomly sub-sample the train to 1 million and test data 250k rows. We follow TabReD and use random sub-sampling. The idea behind this instead of a time-based subsampling is to keep data from various time periods and model the distribution shift across the full time horizon.
""",
    # Same as consumer_complaints.ipynb
    dataset_year="2025",
    domain_str="finance",
    # Data Source
    dataset_source="GOV Website",
    original_dataset_source_download_link="https://www.consumerfinance.gov/data-research/consumer-complaints/",
    download_description="""
We utilize the newest data (acquired on 23/01/2026) from the government website. We use the following commands to download and organize the data:

wget https://files.consumerfinance.gov/ccdb/complaints.csv.zip && unzip complaints.csv.zip && rm complaints.csv.zip
mkdir -p local-data-warehouse/consumer_complaints && mv complaints.csv local-data-warehouse/consumer_complaints
""",
    # References
    academic_reference_bibtex=r"""@misc{cfpb2025ConsumerComplaintDatabase,
  author       = {{Consumer Financial Protection Bureau}},
  title        = {Consumer Complaint Database},
  year         = {2025},
  howpublished = {\url{https://www.consumerfinance.gov/data-research/consumer-complaints/}},
  note         = {Accessed: 2026-01-23},
}
""",
    academic_reference_bibtex_key="cfpb2025ConsumerComplaintDatabase",
    license="U.S. Government Works",
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
The dataset on Kaggle (https://www.kaggle.com/datasets/selener/consumer-complaint-database) has data from up until 2019. We use the newest version from the government website with data up until 2025.

- Context and descriptions of the features can be found here: https://cfpb.github.io/api/ccdb/fields.html
- "Company public response" is not free text but selected "from a set list of options".
- Following TexTabBench, we focus on predicting the type of closure a complaint got, that is the "Company response to consumer" column. We want to predict if a complaint will be closed with just an explanation, with non-monetary relief, or with monetary relief.
- We filter the data to only include entries after consumer disputations were discontinued as this represent a shift in protocol. This filters all data before April 24th 2017.
- We filter all cases where the "Consumer consent provided?" is in progress or got an untimely response.
- We filter all rows that do not include consent to share their complaint narrative. This ensures the data contains text sentences.
- We drop "Company public response" as it leaks the target variable.
- We only allow complaints from US states (no territories or international complaints) and remove cases with missing states.
- The data has unresolved spatial information in the ZIP code. Some ZIPs are censored (ending in "XXX" or full removed "XXXXXX").
- We add a feature for "Low population area", which determines that the ZIP is censored.
- The tags filed contains only three non-nan labels, of which one is a duplicate of the others. We created two categorical features from it instead.
- We drop duplicates (2% of the data) as the data should not contain naturally occurring duplicates and this likely results from some overlap in data collection or data entry or faulty re-submissions. We investigated some of the duplicates and they appear to be identical complaints. There exist duplicates with different target labels, we also drop these as we have no way to determine what the correct label is.
- We drop constant columns, the complaint ID, and when the complaint was send to the company (as it does not related to the target task)
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Company response to consumer",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="Company response to consumer",
    time_on="Date received",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "complaints.csv")
print("Loaded data shape:", df.shape)

# Only keep entries after responses were discontinued as this represent a shift in protocol and closure (so data after April 2017)
# - Note this filters one of the classes from TexTabBench completely as it was an "old" label
df = df[df["Consumer disputed?"].isna()]
df = df.drop(columns=["Consumer disputed?"])

# Filter to complaints that include sentences and have consent
df = df[df["Consumer consent provided?"] == "Consent provided"]
df = df.drop(columns=["Consumer consent provided?"])

# Filter to only keep relevant rows that got closed
df = df[df["Company response to consumer"].isin(["Closed with explanation", "Closed with non-monetary relief", "Closed with monetary relief"])]

# Drop leakage variable
df = df.drop(columns=["Timely response?", "Company public response"])

# Remove non-us states
non_state = [
    "AA", "AE", "AP", "AS", "DC",
    "FM", "GU", "MH", "MP", "PR",
    "PW", "VI",
    "UNITED STATES MINOR OUTLYING ISLANDS",
]
df = df[~(df["State"].isin(non_state) | df["State"].isna())]

# if the ZIP ends with XXX, it means the complaints originates from a place with less than 20k people.
df["Low population area"] = "False"
df.loc[(df["ZIP code"].str.contains("XX") & (df["ZIP code"] != "XXXXX")), "Low population area"] = "True"
df.loc[df["ZIP code"] == "XXXXX", "Low population area"] = np.nan

# Resolve Multi-Tags feature into two categorical features
assert ['Older American', 'Older American, Servicemember', 'Servicemember', 'nan'] == list(np.unique(df["Tags"].astype(str)))
tags_nan_mask = df["Tags"].isna()
tags_is_older_american = df["Tags"].str.contains("Older American", na=False)
tags_is_servicemember = df["Tags"].str.contains("Servicemember", na=False)
df["Tag: Older American"] = "False"
df["Tag: Servicemember"] = "False"
df.loc[tags_is_older_american, "Tag: Older American"] = "True"
df.loc[tags_is_servicemember, "Tag: Servicemember"] = "True"
df.loc[tags_nan_mask, ["Tag: Older American", "Tag: Servicemember"]] = np.nan
df = df.drop(columns=["Tags"])

# Drop column
df = df.drop(columns=[
    "Submitted via", # only web after our preprocessing
    "Complaint ID", # we have new-ness of complaint from other columns, so this does not tell anything new
    "Date sent to company", # just shows processing delay from CFPB and not related to the target. Otherwise, almost always identical to "Date received".
])

as_cat_type = [
    "Product",
    # Higher cardinality categorical features, but still cat per definition of the data!
    "Sub-product",
    "Issue",
    "Sub-issue",
    "State",
    "Low population area",
    "Company response to consumer",
    "Tag: Older American",
    "Tag: Servicemember",
]
as_string_type = [
    "Company", # categorical based on feature description but given that we can have new companies in the future, it cannot be a "normal" categorical feature
    "Consumer complaint narrative",
    "ZIP code",
]

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")
df["Date received"] = pd.to_datetime(df["Date received"])

# We drop duplicates as the data should not contain naturally occurring duplicates.
df = df.drop_duplicates(
    subset=df.columns.difference([task_mold.target_column_name])
)

Loaded data shape: (13184632, 18)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 3,386,817
Columns: 13
Use sampling: True (sample size: 338,682)
Get missing and unique counts per column...


missing/unique per-col:   0%|          | 0/13 [00:00<?, ?it/s]

Get example values per column...


examples per-col:   0%|          | 0/13 [00:00<?, ?it/s]

Get cat stats...


...

Get target stats...
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Consumer complaint narrative', 'ZIP code', 'Company', 'Date received', 'Sub-issue', 'Issue', 'Sub-product', 'State', 'Product', 'Low population area']


Rows remaining as candidates after top-10 filter: 595 (of 3,386,817)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...


hash cols:   0%|          | 0/13 [00:00<?, ?it/s]

group check:   0%|          | 0/13 [00:00<?, ?it/s]

Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company,State,ZIP code,Company response to consumer,Low population area,Tag: Older American,Tag: Servicemember
3,2025-10-14,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information is missing that should be on the report,My credit report contains incorrect and misleading items.,"EQUIFAX, INC.",TX,75062,Closed with non-monetary relief,False,NaN,NaN
4,2025-10-26,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Be sure that disputed accounts are taken down promptly in accordance with the law.,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,93619,Closed with non-monetary relief,False,NaN,NaN
25,2020-05-08,"Credit reporting, credit repair services, or other personal consumer reports",Credit reporting,Incorrect information on your report,Information belongs to someone else,These are not my accounts.,Experian Information Solutions Inc.,NV,89030,Closed with explanation,False,NaN,NaN
27,2025-10-21,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,"I do not recognize the aforementioned accounts, collections, or hard inquiries as reported. The aforementioned accounts, collections, and hard inquiries appearing on my consumer credit report maintained by you were not opened, made, or initiated by me. All of the aforementioned accounts, collections, and hard inquiries are the result of identity theft and fraud.","TRANSUNION INTERMEDIATE HOLDINGS, INC.",SC,29455,Closed with non-monetary relief,False,NaN,NaN
28,2025-11-10,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,"I am formally disputing inaccurate information on my credit reports and requesting an identity theft block under the Fair Credit Reporting Act ( 15 U.S.C. 1681c-2 and 1681i ).\n\nThe following accounts, collections, inquiries, and reported late payments are fraudulent, unauthorized, or otherwise inaccurate. I did not open or authorize these accounts. Since they are the result of identity theft or misreporting, they must be blocked and permanently deleted from my credit files. \nDisputed Accounts and Collections 1. XXXX XXXX XXXX XXXX XXXX {$1100.00} Unauthorized. Please block/delete. \n2. XXXX XXXX XXXX {$1200.00} Unauthorized. Please block/delete. \nXXXX. I XXXX XXXX {$4400.00} Unauthorized. Please block/delete. \nXXXX. XXXX XXXX, XXXX {$1100.00} Unauthorized. Please block/delete. \nXXXX. XXXXXXXX XXXX XXXX XXXX {$10000.00} ( Profit & Loss Write-Off ) Unauthorized. Also, 7 late payments are reported on this account. Since the account is unauthorized, the tradeline and payment history must be deleted in full. \nXXXX. XXXX XXXX {$26000.00} Profit & Loss Write-Off Unauthorized. Please block/delete. \nXXXX. XXXX XXXX {$1200.00} ( Charge-Off / Profit & Loss Write-Off ) Unauthorized. Additionally, 10 missed payments are being reported. Since this account is fraudulent, the tradeline and all late payments must be permanently deleted. \nUnauthorized/Improper Hard Inquiries The following hard inquiries were not authorized and must be deleted : XXXX XXXX Under FCRA 1681b, only permissible purpose inquiries may appear on a credit file. These inquiries were not authorized and must be removed.","TRANSUNION INTERMEDIATE HOLDINGS, INC.",NY,10466,Closed with non-monetary relief,False,NaN,NaN


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Tag: Older American,category,3074519.0,90.78,2.0,"False, True"
1,Tag: Servicemember,category,3074519.0,90.78,2.0,"True, False"
2,Sub-issue,category,227662.0,6.72,212.0,"Information belongs to someone else, Reporting company used your report improperly, Their investigation did not fix an error on your report, Account information incorrect, Account status incorrect, Credit inquiries on your report that you don't recognize, Investigation took more than 30 days, Debt is not yours, Was not notified of investigation status or results, Personal information incorrect"
3,Low population area,category,145979.0,4.31,2.0,"False, True"
4,Product,category,0.0,0.00,14.0,"Credit reporting or other personal consumer reports, Credit reporting, credit repair services, or other personal consumer reports, Debt collection, Checking or savings account, Credit card or prepaid card, Mortgage, Money transfer, virtual currency, or money service, Credit card, Vehicle loan or lease, Student loan"
5,Sub-product,category,36.0,0.00,58.0,"Credit reporting, General-purpose credit card or charge card, Checking account, I do not know, Other debt, Credit card debt, Conventional home mortgage, Domestic (US) money transfer, Loan, Medical debt"
6,Issue,category,0.0,0.00,93.0,"Incorrect information on your report, Improper use of your report, Problem with a company's investigation into an existing problem, Problem with a credit reporting company's investigation into an existing problem, Attempts to collect debt not owed, Managing an account, Written notification about debt, Trouble during payment process, Other transaction problem, Problem with a purchase shown on your statement"
7,State,category,0.0,0.00,50.0,"TX, FL, CA, GA, NY, IL, NC, PA, NJ, MD"
8,Company response to consumer,category,0.0,0.00,3.0,"Closed with explanation, Closed with non-monetary relief, Closed with monetary relief"
9,Date received,datetime64[ns],0.0,0.00,3175.0,"2025-01-17 00:00:00, 2025-01-18 00:00:00, 2025-01-15 00:00:00, 2025-01-16 00:00:00, 2025-01-20 00:00:00, 2025-01-23 00:00:00, 2025-01-24 00:00:00, 2025-01-21 00:00:00, 2025-01-22 00:00:00, 2025-08-06 00:00:00"


In [6]:
# Numeric Feature Statistics
numeric_stats

'No numeric features to summarize.'

In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column                       rank                                                                        
Company                      1                                                           EQUIFAX, INC.   
                             2                                  TRANSUNION INTERMEDIATE HOLDINGS, INC.   
                             3                                     Experian Information Solutions Inc.   
                             4                                       CAPITAL ONE FINANCIAL CORPORATION   
                             5                                                    JPMORGAN CHASE & CO.   
Company response to consumer 1                                                 Closed with explanation   
                             2                                         Closed with non-monetary relief   
                             3                                             Closed with monetary relief   
Consumer complaint narrative 1     In accordance with the Fair Credit Reporting act. The List of ac...   
                             2     My credit reports are inaccurate. These inaccuracies are causing...   
                             3     You have reported inaccurate and unauthorized accounts on my cre...   
                             4     Upon reviewing my credit report, I have identified inaccurate ac...   
                             5     I am writing to have the following information removed from my c...   
Date received                1                                                     2025-01-17 00:00:00   
                             2                                                     2025-01-18 00:00:00   
                             3                                                     2025-01-15 00:00:00   
                             4                                                     2025-01-16 00:00:00   
                             5                                                     2025-01-20 00:00:00   
Issue                        1                                    Incorrect information on your report   
                             2                                             Improper use of your report   
                             3         Problem with a company's investigation into an existing problem   
                             4     Problem with a credit reporting company's investigation into an ...   
                             5                                       Attempts to collect debt not owed   
Low population area          1                                                                   False   
                             2                                                                    True   
                             3                                                                    <NA>   
Product                      1                     Credit reporting or other personal consumer reports   
                             2     Credit reporting, credit repair services, or other personal cons...   
                             3                                                         Debt collection   
                             4                                             Checking or savings account   
                             5                                             Credit card or prepaid card   
State                        1                                                                      TX   
                             2                                                                      FL   
                             3                                                                      CA   
                             4                                                                      GA   
                             5                                                                      NY   
Sub-issue                    1                                     In

In [8]:
# Target Distribution
target_df

,count,pct
Company response to consumer,,
Closed with explanation,2226089,65.73
Closed with non-monetary relief,1083579,31.99
Closed with monetary relief,77149,2.28


## Task Curation

In [9]:
# Filter to year 2025
df_2025 = df[df[task_mold.time_on].dt.year == 2025].copy()

# Create a year-month column for grouping
df_2025["year_month"] = df_2025[task_mold.time_on].dt.to_period("M")

# 1) Total number of samples per month
monthly_totals = (
    df_2025
    .groupby("year_month")
    .size()
    .rename("total_samples")
)

# 2) Count of each class per month
monthly_class_counts = (
    df_2025
    .groupby(["year_month", task_mold.target_column_name])
    .size()
    .unstack(fill_value=0)
)

# Optional: combine both into a single DataFrame
result = monthly_class_counts.join(monthly_totals)

# Optional: sort by month and convert PeriodIndex to timestamp (month start)
result = result.sort_index()
result.index = result.index.to_timestamp()

/tmp/ipykernel_611303/3372607303.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["year_month", task_mold.target_column_name])


In [10]:
result

,Closed with explanation,Closed with monetary relief,Closed with non-monetary relief,total_samples
year_month,,,,
2025-01-01,120288,1560,43667,165515
2025-02-01,54026,1074,32450,87550
2025-03-01,56762,1127,41130,99019
2025-04-01,56442,1090,41407,98939
2025-05-01,59634,1224,41165,102023
2025-06-01,60198,1295,46249,107742
2025-07-01,69508,1442,48707,119657
2025-08-01,72032,1330,30270,103632
2025-09-01,53226,1155,23678,78059


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import subsample_temporal

# Sort by time
df = df.sort_values(by=task_mold.time_on).reset_index(drop=True)
split_time = pd.Timestamp("2025-09-01")

test_idx = df.index[
    df[task_mold.time_on] >= split_time
].to_numpy().tolist()
train_idx = df.index[
    df[task_mold.time_on] < split_time
].to_numpy().tolist()

df, train_idx, test_idx = subsample_temporal(
    df=df,
    train_idx=train_idx,
    test_idx=test_idx,
    stratify_on=task_mold.stratify_on,
)

# Size and class distribution checks
print("Train size:", len(train_idx), " | Test size:", len(test_idx))
print("Train target distribution:\n", df.loc[train_idx, task_mold.target_column_name].value_counts(normalize=True))
print("Test target distribution:\n", df.loc[test_idx, task_mold.target_column_name].value_counts(normalize=True))

splits = {
    0: {
        0: (train_idx, test_idx),
    }
}

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="""We try to create splits that simulate a model deployed to solve the task.

- The official data is updated daily but companies have 180 days to respond to a new complaint.
- We could simulate a model that is refitted daily, but this would need many splits. Moreover, data from just a few days is likely not enough to create a robust test set.
- Thus, we instead simulate a model that is refit every three months and then deployed.
- This introduces the unrealistic downside of data shift across a month that would not exist in a real-world model.

We create one test split by using all data after 2025-09-0101 as test splits. This represents a refit horizon of ca. 3 months.
We use all data before as training data.
""",
    splits=splits,
    time_horizon=3,
    time_horizon_unit="months",
)

Train size: 1000000  | Test size: 226140
Train target distribution:
 Company response to consumer
Closed with explanation            0.656952
Closed with non-monetary relief    0.319809
Closed with monetary relief        0.023239
Name: proportion, dtype: float64
Test target distribution:
 Company response to consumer
Closed with explanation            0.661882
Closed with non-monetary relief    0.321770
Closed with monetary relief        0.016348
Name: proportion, dtype: float64


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to consumer_complaints/versions/019d738b-4c6e-751f-972a-5f63b1508f70


019d738b-4c6e-751f-972a-5f63b1508f70
930fa0fa1d318d925f32ae44532b5259337d698a956c73315aaf9f5c6047a2c1
